# P17: working agent for customer-support


* import modules

In [ ]:
from langchain.tools import tool
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage,AIMessage, HumanMessage
from langchain_core.messages.tool import ToolMessage
from langgraph.graph import StateGraph


* The @tool decorator does two important things:

1-Registers the function as a callable tool with a name (cancel_order) and a schema derived from the type hints.

2-Exposes the docstring as the tool description — the LLM reads this to decide when to call the tool.

In [ ]:
#1: Define our single business tool
@tool
def cancel_order(order_id:str) -> str:
    """Cancel an order that hasn't shipped."""
    return f"Order {order_id} has been cancelled."

cancel_order.invoke("A12345")
# → "Order A12345 has been cancelled."

In [19]:
#2: The agent "brain": invoke LLm, run tool, then invoke LLM again
def call_model(state):
    msgs = state["messages"]
    order = state.get("order", {"order_id":"UNKNOWN"})

    #sytem prompt tells the model exactly what to do
    prompt = (
        f'''You are an ecommerce support agent.
        ORDER ID: {order['order_id']}
        If the customer asks to cancel, call cancel_order(order_id)
        and then send a simple confirmation.
        Otherwise, just respond normally.'''
    )
    full = [SystemMessage(prompt)] + msgs

    #1st LLM pass: decides wether to call our tool
    first = ChatOllama(model="llama3.1",temperature=0).invoke(full)
    out=[first]

    if getattr(first, "tool_calls", None):
        #run the cancel_order tool
        tc = first.tool_calls[0]
        result =cancel_order(**tc["args"])
        out.append(ToolMessage(content=result, tool_call_id=tc["id"]))
        #2nd LLM pass: generate the final confirmation text
        second = ChatOllama(model="llama3.1",temperature=0)(full + out)
        out.append(second)
    return {"messages": out}

In [20]:
from typing import TypedDict, List
from langchain_core.messages import BaseMessage

class AgentState(TypedDict):
    order: dict
    messages: List[BaseMessage]

def construct_graph():
    graph = StateGraph(AgentState)  # ← pass the class, not a dict
    graph.add_node("assistant", call_model)
    graph.set_entry_point("assistant")
    return graph.compile()

graph = construct_graph()

In [22]:
if __name__ == "__main__":
    example_order = {"order_id": "A12345"}
    convo = [HumanMessage(content="Hi, I want to cancel my order A12345")]
    result=graph.invoke({"order": example_order, "messages": convo})
    for msg in result["messages"]:
        print(f"{msg.type}: {msg.content}")

ai: I'd be happy to help you with that.

However, before we proceed, could you please confirm that you would like to cancel the entire order? This will result in a full refund and cancellation of all items associated with Order ID A12345. 

Please let me know if this is what you're looking for.
